#### Download all tables in gold layers to csv

#####1. Copy tables to temp dbfs

In [0]:
%python
#Set Schema name and output dbfs folder
schema_name = "gold_olist"
output_dbfs_base = "dbfs:/tmp/gold_olist_tables"

# clean old folder if exists
dbutils.fs.rm(output_dbfs_base, recurse=True)
dbutils.fs.mkdirs(output_dbfs_base)

print("DBFS folder ready:", output_dbfs_base)



In [0]:
%python
#List all tables in gold_schema: 
tables = [t.name for t in spark.catalog.listTables(schema_name)]
print(f"Tables found in {schema_name}:", tables)


In [0]:
%python
#Loop through all tables and write to dbfs
for table in tables:
    print("Exporting:", table)
    
    # Load the table
    df = spark.table(f"{schema_name}.{table}")
    
    # Set DBFS output folder for this table
    table_dbfs_dir = f"{output_dbfs_base}/{table}"
    
    # Save as CSV (one file with header)
    df.coalesce(1).write.mode("overwrite").option("header", "true").csv(table_dbfs_dir)
    
    csv_part = [f.path for f in dbutils.fs.ls(table_dbfs_dir) if f.name.endswith(".csv")][0]
    dbutils.fs.cp(csv_part, f"dbfs:/FileStore/gold_olist_tables/{table}.csv")
    
    print(f"{schema_name}.{table} saved --> dbfs:/FileStore/gold_olist_tables/{table}.csv")


In [0]:
%python
# Check dbfs csv folder in FileStore folder: 
display(dbutils.fs.ls('dbfs:/FileStore/gold_olist_tables/'))



##### 2. Download csv files

In [0]:
%python
import shutil
import os

# local driver temp folder
tmp_local = "/tmp/gold_olist_csvs_zip"
# remove folder if it exists
if os.path.exists(tmp_local):
    shutil.rmtree(tmp_local)

# recreate it
os.makedirs(tmp_local, exist_ok=True)

# copy all CSVs from DBFS to local driver
for table in tables:
    dbutils.fs.cp(f"dbfs:/FileStore/gold_olist_tables/{table}.csv",
                  f"file:{tmp_local}/{table}.csv")

# zip them
shutil.make_archive("/tmp/gold_olist_tables", 'zip', tmp_local)

# copy zip to FileStore
dbutils.fs.cp("file:/tmp/gold_olist_tables.zip", "dbfs:/FileStore/gold_olist_tables.zip")


Then, the zip file is downloaded in this link: https://<workspace>.azuredatabricks.net/files/gold_olist_tables.zip